# Lab 4: AgentCore Browser를 활용한 예방 에이전트

## 개요
AgentCore Browser를 사용하여 AWS 문서를 실시간으로 조사하고 선제적인 인프라 권장 사항을 제공하는 Strands 기반 예방 에이전트를 구축합니다.

## 목표
- AgentCore Browser가 통합된 Strands 에이전트 생성
- 예방 중심의 분석 워크플로 구현
- AWS 모범 사례를 실시간으로 조사하는 도구 구축
- 인프라 문제 예방 시나리오로 에이전트 테스트
- 문제를 방지하기 위한 선제적 권장 사항 생성

## 학습 내용
- AgentCore Browser를 Strands 에이전트와 통합하는 방법
- 사후 대응형 문제 해결과 대비되는 예방 중심 워크플로를 구현하는 방법
- 브라우저 자동화를 사용하여 최신 AWS 모범 사례를 조사하는 방법
- 에이전트의 예방 분석 및 권장 사항 패턴

## 아키텍처 개요
```
┌─────────────────┐    ┌──────────────────────┐    ┌─────────────────────┐
│   User Request  │───▶│  Strands Agent       │───▶│  AgentCore Browser  │
│   (Prevention)  │    │  (Prevention)        │    │                     │
└─────────────────┘    └──────────────────────┘    └─────────────────────┘
                                │                            │
                                ▼                            ▼
                       ┌──────────────────────┐    ┌─────────────────────┐
                       │  Prevention Analysis │    │  Real-time Web      │
                       │  ├─ Risk Assessment  │    │  Research           │
                       │  ├─ Best Practices   │    │  ├─ AWS Docs        │
                       │  └─ Recommendations  │    │  ├─ Best Practices   │
                       └──────────────────────┘    │  └─ Current Standards│
                                │                  └─────────────────────┘
                                ▼                            │
                       ┌──────────────────────┐              ▼
                       │  Proactive           │    ┌─────────────────────┐
                       │  Recommendations     │◀───│  Documentation      │
                       │  (Before Issues)     │    │  Analysis & Insights│
                       └──────────────────────┘    └─────────────────────┘
```

**주요 구성 요소:**
- **예방 중심**: 사후 대응형 수정이 아닌 선제적 권장 사항 제공
- **실시간 조사**: 최신 AWS 문서에 실시간으로 접근
- **브라우저 통합**: AgentCore Browser를 통한 웹 콘텐츠 접근
- **최신 표준**: 항상 최신 AWS 모범 사례 사용

## 0. 필수 패키지 설치

모든 종속 항목이 설치되도록 이 셀을 먼저 실행합니다.

In [ ]:
%pip install -q -r requirements.txt
print("✅ Prevention agent dependencies installed")

## 1. 필수 모듈 가져오기

In [ ]:
# AWS SDK 및 구성
import logging
from datetime import datetime

# Strands 프레임워크
from strands import Agent
from strands.models import BedrockModel
from strands.tools import tool
from strands_tools.browser import AgentCoreBrowser

# 워크숍 구성
# Bedrock AgentCore Starter Toolkit
from bedrock_agentcore_starter_toolkit import Runtime
from lab_helpers.config import MODEL_ID, AWS_REGION, AWS_PROFILE, WORKSHOP_NAME
from lab_helpers.parameter_store import get_parameter, put_parameter
from lab_helpers.constants import PARAMETER_PATHS
from lab_helpers.lab_04.gateway_setup import AgentCoreGatewaySetup

# Notebook 로깅 구성
logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
logger = logging.getLogger(__name__)

print("✅ Imports loaded")
print(f"   Workshop: {WORKSHOP_NAME}")
print(f"   Region: {AWS_REGION}")
print(f"   Model: {MODEL_ID}")
print("🌐 AgentCore Browser tool imported")
print("🔮 Prevention agent ready for setup")

## 2. 사전 요구 사항 확인

In [ ]:
# 사전 요구 사항을 사용할 수 있는지 확인
try:
    # AWS 자격 증명 테스트
    import boto3

    sts_client = boto3.client("sts", region_name=AWS_REGION)
    identity = sts_client.get_caller_identity()
    account_id = identity["Account"]

    # AgentCore Browser 사용 가능 여부 테스트
    browser_tool = AgentCoreBrowser(region=AWS_REGION)

    print(f"✅ Prerequisites verified: AWS Account {account_id}, AgentCore Browser available")
    print(f"   Region: {AWS_REGION}")
    print(f"   Profile: {AWS_PROFILE}")
    print(f"   Model ID: {MODEL_ID}")
    print(f"   Identity: {identity.get('Arn', 'Unknown')}")
    print("   Browser Tool: AgentCoreBrowser initialized")

except Exception as e:
    print(f"❌ Error: {e}")
    print("Please ensure AWS credentials are configured and AgentCore Browser permissions are available.")

## 3. 브라우저가 통합된 예방 에이전트 생성

**목표:** 실시간 웹 조사를 위해 AgentCore Browser가 통합된 Strands 에이전트를 설정합니다.

**접근 방식:** 브라우저 도구를 초기화하고 예방 중심의 시스템 프롬프트를 사용하는 에이전트를 생성합니다.

**핵심 학습 내용:** AgentCore Browser를 활용하여 문제 예방을 위한 문서를 실시간으로 조사하는 방법을 알아봅니다.

In [ ]:
### 3.1: 브라우저 도구로 예방 에이전트 설정


def setup_prevention_agent(region=AWS_REGION):
    """AgentCore Browser 도구로 Strands 에이전트를 설정합니다."""
    try:
        # strands_tools.browser 패턴에 따라 Browser 도구 초기화
        browser_tool = AgentCoreBrowser(region=region)

        # Bedrock 모델 설정
        model = BedrockModel(
            model_id=MODEL_ID,
            streaming=True,
        )

        # 브라우저 도구를 사용하는 에이전트 생성
        agent = Agent(model=model, tools=[browser_tool.browser])

        logger.info("✅ SRE Prevention Agent ready with AgentCore Browser")
        logger.info(f"🌍 Region: {region}")
        logger.info("🌐 Browser tool: AgentCoreBrowser integrated")

        return agent

    except Exception as e:
        logger.error(f"❌ Failed to setup agent: {e}")
        return None


# 에이전트 설정
agent = setup_prevention_agent()
if agent:
    print("✅ Strands prevention agent created successfully")
    print(f"   Model: {MODEL_ID}")
    print("   Tools: 1 (AgentCore Browser)")
    print("🌐 AgentCore Browser integration ready")
    print("🔮 Ready for prevention analysis")
else:
    print("❌ Agent setup failed!")
    print("   Check AgentCore Browser initialization and AWS credentials")

## 4. 브라우저 통합 테스트

**목표:** 기본 웹 접근 테스트를 통해 AgentCore Browser 기능을 확인합니다.

**접근 방식:** AWS 홈페이지에 접속하여 브라우저 도구를 테스트합니다.

**핵심 학습 내용:** 브라우저 연결 및 콘텐츠 추출을 검증하는 방법을 알아봅니다.

In [ ]:
### 4.1: 브라우저 통합 테스트

# 에이전트 설정

agent = setup_prevention_agent()


def test_browser_integration(agent):
    """AgentCore Browser 통합을 테스트합니다."""
    if not agent:
        print("❌ Agent not available for testing")
        return False

    print("🧪 Testing AgentCore Browser Integration...")
    print("=" * 50)

    test_prompt = """
    Please use the browser tool to visit https://aws.amazon.com and get the page title
    
    
    This is just a test to verify the browser tool is working correctly.
    """

    try:
        start_time = datetime.now()
        response = agent(test_prompt)
        test_time = (datetime.now() - start_time).total_seconds()

        print(f"✅ Browser test completed in {test_time:.2f} seconds")

        # 응답 표시(너무 길면 일부 생략)
        response_content = response.message.get("content", [])
        if response_content:
            for content in response_content:
                if isinstance(content, dict) and "text" in content:
                    text = content["text"]
                    if len(text) > 800:
                        print("\n📋 BROWSER TEST RESULTS (first 800 chars):")
                        print("-" * 30)
                        print(f"{text[:800]}...\n\n[Full output: {len(text)} chars]")
                    else:
                        print("\n📋 BROWSER TEST RESULTS:")
                        print("-" * 30)
                        print(text)
                elif isinstance(content, str):
                    print("\n📋 BROWSER TEST RESULTS:")
                    print("-" * 30)
                    print(content)

        return True

    except Exception as e:
        print(f"❌ Browser test failed: {e}")
        return False


# 브라우저 테스트 실행
if agent:
    print("💡 To run browser test, uncomment the line below:")
    test_result = test_browser_integration(agent)
    if test_result:
        print("\n🎉 AgentCore Browser integration test PASSED!")
    else:
        print("\n❌ AgentCore Browser integration test FAILED!")
    print("⚠️  Note: This will make a real web request to AWS homepage")
else:
    print("❌ Agent not available for browser testing")

## Memory의 정보로 컨텍스트 보강

선별하여 저장한 Memory에서 추가 정보를 가져와 진단 정보로 컨텍스트를 보강합니다.

In [ ]:
agent_memory_client = boto3.client("bedrock-agentcore", region_name=AWS_REGION)

memory_id = get_parameter(PARAMETER_PATHS["memory"]["memory_id"])
memory_session_id = get_parameter(PARAMETER_PATHS["memory"]["default_session_id"])

print(memory_id)
print(memory_session_id)
actor_id = "diagnostics_agent"


# 쓰기가 성공했는지 확인하기 위해 에이전트 Memory에 추가된 이벤트 나열
params = {
    "memoryId": memory_id,
    "actorId": actor_id,
    "sessionId": memory_session_id,
    "includePayloads": True,
}
# 모든 메시지 가져오기
response = agent_memory_client.list_events(**params)
additional_context = ""
for event in response.get("events", []):
    payload = event.get("payload", [])
    for i, item in enumerate(payload):
        if "conversational" in item:
            text = item["conversational"]["content"]["text"]
            additional_context += text
additional_context

## 5. 예방 분석 워크플로

**목표:** 실제 AWS 문서 조사를 활용한 예방 분석 워크플로를 살펴봅니다.

**접근 방식:** 브라우저로 모범 사례를 조사하여 종합적인 예방 분석을 실행합니다.

**핵심 학습 내용:** 예방 에이전트가 선제적 조치를 조사하고 권장하는 방법을 알아봅니다.

In [ ]:
### 5.1: 예방 분석 실행

# 에이전트 설정

agent = setup_prevention_agent()


def analyze_prevention_opportunities(agent):
    """인프라를 분석해 예방 기회를 찾습니다."""
    if not agent:
        logger.error("❌ Agent not initialized")
        return None

    logger.info("🔮 Starting prevention opportunity analysis...")

    prevention_prompt = f"""
    I need you to analyze our CRM infrastructure for prevention opportunities using the browser tool to access AWS documentation. 
    Our CRM application is hosted on EC2 instances with a DynamoDB backend for customer data. 

    Here is additional information about the issues identified and fixes that have been applied:

    {additional_context}
    
    Please use the browser tool to access these specific AWS documentation pages and provide analysis:
    
    1. First, use the browser tool to visit: https://docs.aws.amazon.com/amazondynamodb/latest/developerguide/best-practices.html
    2. Then visit: https://docs.aws.amazon.com/AWSEC2/latest/UserGuide/ec2-best-practices.html  
    3. Finally visit: https://docs.aws.amazon.com/wellarchitected/latest/framework/
    
    Based on what you find in the AWS documentation, provide analysis focusing on:
    
    1. **Proactive Infrastructure Management**: Best practices we should implement
    2. **Customer Impact Prevention**: How to prevent issues that could affect customer experience  
    3. **Performance Optimization**: Latest AWS recommendations for EC2 and DynamoDB optimization
    4. **Monitoring and Alerting**: Best practices for proactive monitoring
    
    Provide your analysis with:
    - Executive summary of prevention opportunities
    - Customer impact prioritization  
    - Implementation roadmap with AWS best practices
    - Success metrics for measuring prevention effectiveness
    
    Use the browser tool to get the most current AWS documentation and recommendations.
    """

    try:
        start_time = datetime.now()
        response = agent(prevention_prompt)
        analysis_time = (datetime.now() - start_time).total_seconds()

        logger.info(f"✅ Prevention analysis completed in {analysis_time:.2f} seconds")

        return {
            "analysis_time": analysis_time,
            "response": response,
            "timestamp": datetime.now().isoformat(),
        }

    except Exception as e:
        logger.error(f"❌ Prevention analysis failed: {e}")
        return None


# 전체 예방 분석 실행
if agent:
    print("🚀 Starting Complete Prevention Analysis Workflow...")
    print("=" * 60)
    print("🔮 This will research current AWS best practices using the browser")
    print("📚 Accessing live AWS documentation for latest recommendations")
    print("⏳ This may take a few minutes as we browse multiple documentation pages...")
    result = analyze_prevention_opportunities(agent)

    if result:
        print("\n🎯 PREVENTION ANALYSIS RESULTS:")
        print(f"Analysis Time: {result['analysis_time']:.2f} seconds")
        print(f"Timestamp: {result['timestamp']}")

        # 에이전트 응답 표시
        response_content = result["response"].message.get("content", [])
        if response_content:
            for content in response_content:
                if isinstance(content, dict) and "text" in content:
                    text = content["text"]
                    if len(text) > 2000:
                        print(f"\n📋 PREVENTION ANALYSIS (first 2000 chars):\n{text[:2000]}...")
                    else:
                        print(f"\n📋 PREVENTION ANALYSIS:\n{text}")
                elif isinstance(content, str):
                    print(f"\n📋 PREVENTION ANALYSIS:\n{content}")

        print("\n🎉 Prevention analysis workflow completed successfully!")
    else:
        print("❌ Prevention analysis failed!")

    print("⚠️  Note: This will make multiple browser requests and may take several minutes")
    print("🔒 This demonstrates proactive analysis vs reactive remediation")

else:
    print("❌ Agent not available for prevention analysis!")

## 6. AgentCore Runtime에 배포

**목표:** 서버리스 실행을 위해 예방 에이전트를 Amazon Bedrock AgentCore Runtime에 배포합니다.

**접근 방식:** AgentCoreRuntimeDeployer를 사용하여 Runtime 인프라를 생성하고 에이전트를 배포합니다.

**핵심 학습 내용:** 브라우저가 통합된 Strands 에이전트를 프로덕션 수준의 서버리스 인프라에 배포하는 방법을 알아봅니다.

### 6.1: Runtime 배포 도구 초기화 및 IAM 역할 생성

In [ ]:
from lab_helpers.lab_04 import AgentCoreRuntimeDeployer, store_runtime_configuration

# 배포 도구 초기화
deployer = AgentCoreRuntimeDeployer(region=AWS_REGION, prefix=WORKSHOP_NAME, verbose=True)

# 사전 요구 사항 확인
print("✓ Checking prerequisites...")
if deployer.check_prerequisites():
    print("✅ All prerequisites met")
else:
    print("❌ Prerequisites not met")
    print("Please install: pip install fastmcp bedrock-agentcore-starter-toolkit strands-agents-tools")

# Browser 권한이 있는 IAM 역할 생성
print("\n🔐 Creating IAM role for Runtime...")
role_info = deployer.create_runtime_iam_role()
print(f"   Role ARN: {role_info['role_arn']}")

### 6.2: AgentCore Runtime에 배포

In [ ]:
### 6.2 a: 배포용 에이전트 코드

# 에이전트 코드 파일 읽기
with open("lab_helpers/lab_04/runtime_mcp_agent_code.py", "r") as f:
    agentcore_agent_code = f.read()

print("✅ Agent code loaded from: lab_helpers/lab_04/runtime_mcp_agent_code.py")
print(f"✓ Loaded agent code: {len(agentcore_agent_code)} bytes")
print(f"✓ Code size: {len(agentcore_agent_code) / 1024:.1f} KB")
print(f"✓ AWS_REGION: {AWS_REGION}")

In [ ]:
### 6.2 b: 에이전트 코드를 디스크에 쓰기
with open("agent-prevention.py", "w") as f:
    f.write(agentcore_agent_code)

print("✅ Agent code written: agent-prevention.py")

### 6.3 JWT Authorizer로 Runtime 구성

**runtime.configure()의 기능:**
- 에이전트 코드와 종속 항목 검증
- Dockerfile 및 AWS 구성 파일 생성
- 배포 청사진 준비(로컬 작업이며 아직 AWS 리소스는 생성되지 않음)
- 실행 역할 및 토큰 검증 설정

**JWT Authorizer 구성:**
`authorizer_configuration` 파라미터는 수신 토큰을 검증하는 방법을 Runtime에 지정합니다.
- **discoveryUrl**: Runtime이 서명 검증용 공개 키를 가져오는 Cognito OIDC 엔드포인트
- **allowedClients**: User Auth 클라이언트(직접 접속 사용자)와 M2M 클라이언트(Gateway)를 모두 허용

**Runtime 자동 동작:**
```
Bearer token in request → Validate signature → Check issuer (Cognito) → Verify client ID in allowedClients → Allow or Reject
```

In [ ]:
### 6.3 a: Cognito 구성 가져오기
from lab_helpers.parameter_store import get_parameter
from lab_helpers.constants import PARAMETER_PATHS

# Lab-01의 SSM Parameter Store에서 Cognito 구성 가져오기
cognito_domain = get_parameter(PARAMETER_PATHS["cognito"]["domain"])
user_pool_id = get_parameter(PARAMETER_PATHS["cognito"]["user_pool_id"])
m2m_client_id = get_parameter(PARAMETER_PATHS["cognito"]["m2m_client_id"])
user_auth_client_id = get_parameter(PARAMETER_PATHS["cognito"]["user_auth_client_id"])

# Cognito 검색 URL 구성 - Runtime은 토큰 검증용 공개 키를 가져올 때 이 URL을 사용
# 직접 구성하는 것보다 안정적인 Parameter Store의 Cognito 도메인 사용
discovery_url = f"https://cognito-idp.{AWS_REGION}.amazonaws.com/{user_pool_id}/.well-known/openid-configuration"

print("✅ Cognito configuration retrieved")
print(f"   Discovery URL: {discovery_url}")
print("   Allowed Clients: User Auth + M2M")
print(f"   User Auth Client ID: {user_auth_client_id}")
print(f"   M2M Client ID: {m2m_client_id}")

In [ ]:
### 6.3 b: JWT Authorizer로 Runtime 구성

# Runtime 객체 초기화
runtime = Runtime()

# JWT Authorizer 구성
# - discoveryUrl: Cognito OIDC 엔드포인트(Runtime이 공개 키를 자동으로 가져옴)
# - allowedClients: User Auth 및 M2M 클라이언트 모두 Runtime 호출 가능
authorizer_config = {
    "customJWTAuthorizer": {
        "discoveryUrl": discovery_url,
        "allowedClients": [user_auth_client_id, m2m_client_id],
    }
}

# 중요: JWT 토큰 검증을 사용하도록 Runtime 구성
runtime.configure(
    entrypoint="agent-prevention.py",
    execution_role=role_info["role_arn"],
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region=AWS_REGION,
    agent_name=f"{WORKSHOP_NAME}_prevention_runtime",
    protocol="MCP",
    authorizer_configuration=authorizer_config,  # ← token validation 활성화
)

print("✅ Runtime configured with JWT authorizer")
print("   Protocol: MCP")
print("   JWT Token Validation: ENABLED")
print("   Allowed Tokens: User Auth + M2M")

### 6.4: Runtime을 AgentCore에 시작

Python SDK의 `runtime.launch()`를 사용하여 구성된 Runtime을 AgentCore에 배포합니다. 이 핵심 단계에서는 로컬 에이전트를 프로덕션 서버리스 서비스로 전환합니다.

#### 6.4 a: 시작 프로세스 이해

  **runtime.launch() 실행 중 수행되는 작업:**

  1️⃣ CodeBuild가 Docker 컨테이너 빌드를 시작합니다.

  2️⃣ requirements.txt의 종속 항목을 설치합니다.

  3️⃣ 이미지를 자동 생성된 Amazon ECR에 푸시합니다.

  4️⃣ AgentCore가 Runtime을 MCP 서비스로 등록합니다.
  
  5️⃣ CloudWatch 로깅을 구성합니다.

  ⏱️ **일반적인 소요 시간:** 5~10분

In [ ]:
# 모든 셀 실행 시나리오를 처리하기 위해 추가
import time

time.sleep(10)

In [ ]:
### 6.4 a: runtime.launch() 실행

print("\n🚀 Launching Runtime to AgentCore...\n")

try:
    # 충돌 시 자동 업데이트를 사용하여 Runtime 시작
    # 동기 방식으로 실행되며 CodeBuild 및 초기 상태 확인이 끝날 때까지 대기
    launch_result = runtime.launch(auto_update_on_conflict=True)

    # 배포 ARN 추출
    runtime_arn = launch_result.agent_arn

    print("✅ Runtime launched successfully!")
    print(f"   Runtime ARN: {runtime_arn}")
    print("\n📝 Configuration stored for next sections:")

except Exception as e:
    print(f"❌ Launch failed: {e}")
    print("\nTroubleshooting:")
    print("  • Check CodeBuild service limits")
    print("  • Verify ECR permissions in IAM role")
    print("  • Review CloudWatch logs for build errors")
    print("  • Ensure all dependencies in requirements.txt are correct")
    raise

In [ ]:
### 6.4 b: Runtime 구성 저장

# Runtime 구성 추출 및 저장
runtime_arn = launch_result.agent_arn
runtime_id = getattr(launch_result, "agent_id", None)

# 구성 저장

store_runtime_configuration(runtime_arn, runtime_id, region=AWS_REGION, prefix=WORKSHOP_NAME)

print("\n✅ Runtime deployed and configured")
print(f"   ARN: {runtime_arn}")
print("   Ready for Gateway registration")

## 7. CloudWatch 로깅 구성

**목표:** Runtime 컨테이너 로그를 위한 CloudWatch Logs 전송을 설정합니다.

**접근 방식:** configure_runtime_logging을 사용하여 로그 전송을 자동으로 설정합니다.

**핵심 학습 내용:** AgentCore를 CloudWatch와 통합하여 관찰성을 확보하는 방법을 알아봅니다.

In [ ]:
from lab_helpers.lab_04.configure_logging import configure_runtime_logging

# CloudWatch Logs 전송 구성
print("📊 Configuring CloudWatch Logs Delivery...")

logging_config = configure_runtime_logging(
    runtime_arn=runtime_arn,
    runtime_id=runtime_id,
    region=AWS_REGION,
    log_type="APPLICATION_LOGS",
)

print("\n✅ Logging configured:")
print(f"   Log Group: {logging_config['log_group_name']}")
print(f"   Status: {logging_config['delivery_status']}")

## 8. AgentCore Gateway 배포

**목표:** Gateway 인프라를 생성하고 Runtime을 대상으로 등록합니다.

**접근 방식:** Cognito JWT 인증을 사용하는 Gateway를 생성한 다음 OAuth2 M2M 자격 증명으로 Runtime을 등록합니다.

**핵심 학습 내용:** 이중 인증 계층(수신 JWT, 송신 OAuth2 M2M)을 설정하는 방법을 알아봅니다.

In [ ]:
# Lab-01에서 Cognito 자격 증명 가져오기
user_auth_client_id = get_parameter(PARAMETER_PATHS["cognito"]["user_auth_client_id"])
user_pool_id = get_parameter(PARAMETER_PATHS["cognito"]["user_pool_id"])

# Cognito OIDC 검색 URL 구성
discovery_url = f"https://cognito-idp.{AWS_REGION}.amazonaws.com/{user_pool_id}/.well-known/openid-configuration"

# 헬퍼를 사용하여 Gateway IAM 역할 생성
gateway_setup = AgentCoreGatewaySetup(region=AWS_REGION, prefix=WORKSHOP_NAME, verbose=False)
role_info = gateway_setup.create_gateway_service_role()
role_arn = role_info["role_arn"]

# boto3로 Gateway 직접 생성(간단한 API 호출)
agentcore = boto3.client("bedrock-agentcore-control", region_name=AWS_REGION)

gateway_response = agentcore.create_gateway(
    name="aiml301-prevention-gateway",
    roleArn=role_arn,
    protocolType="MCP",
    authorizerType="CUSTOM_JWT",
    authorizerConfiguration={
        "customJWTAuthorizer": {
            "discoveryUrl": discovery_url,
            "allowedClients": [user_auth_client_id],
        }
    },
)

gateway_id = gateway_response["gatewayId"]
gateway_url = gateway_response["gatewayUrl"]

# 구성 저장
put_parameter(PARAMETER_PATHS["lab_04"]["gateway_id"], gateway_id)
put_parameter(PARAMETER_PATHS["lab_04"]["gateway_role_arn"], role_arn)

print("✅ Gateway deployed with JWT authorization")
print(f"   Gateway ID: {gateway_id}")
print(f"   Gateway URL: {gateway_url}")
print("   Auth Type: Cognito JWT")

### 8.1: M2M 인증을 사용하여 Runtime을 Gateway 대상으로 추가

  **목표:** OAuth2 M2M 인증을 사용하여 Runtime을 Gateway 대상으로 등록합니다.

  **아키텍처:**

```
  User (JWT) → Gateway (JWT Validation) → Runtime (M2M Token)
                      ↓
              Gateway automatically:
              - Calls GetResourceOauth2Token
              - Retrieves credentials from Secrets Manager
              - Gets M2M access token from Cognito
              - Injects Bearer token in request
                      ↓
              Calls Runtime with Authorization: Bearer {token}
```
  **Gateway가 M2M OAuth2 토큰을 자동으로 전송하는 방식:**

  1. **Credential Provider 저장소**
     - OAuth2 credential provider는 `clientId` + `clientSecret`을 AWS Secrets Manager에 암호화하여 저장합니다.
     - Provider ARN은 이 보안 저장 위치를 가리킵니다.

  2. **자동 토큰 검색 흐름**
  ```
     Gateway target created with credentialProviderConfigurations
         ↓
     Gateway needs to call Runtime target
         ↓
     Gateway calls GetResourceOauth2Token API (automatic, built-in)
         ↓
     AgentCore Identity retrieves client_id + client_secret from Secrets Manager
         ↓
     AgentCore calls Cognito token endpoint:
     POST /token
     Body: grant_type=client_credentials&client_id=...&client_secret=...&scope=...
         ↓
     Cognito returns M2M access token
         ↓
     AgentCore caches token + manages refresh lifecycle
         ↓
     Gateway injects token in request header:
     Authorization: Bearer {access_token}
         ↓
     Gateway calls Runtime MCP endpoint with Bearer token
         ↓
     Runtime validates JWT signature using Cognito public keys (JWKS)
```
  3. **사용자 지정 코드 불필요**
  - 모든 자격 증명 관리는 자동으로 수행됩니다.
  - 토큰 갱신은 자동으로 수행됩니다.
  - 토큰 삽입은 자동으로 수행됩니다.
  - 코드에서는 credential provider ARN만 지정하면 됩니다.

  **핵심 학습 내용:** 사용자 기반 수신 인증(JWT)과 서비스 기반 송신 인증(OAuth2 M2M)으로 구성된 이중 인증을 알아봅니다.

In [ ]:
# 모든 셀 실행 시나리오를 처리하기 위해 추가
import time

time.sleep(10)

Lab-03을 성공적으로 실행했다면 해당 Lab에서 생성한 credentials provider를 재사용합니다. Lab-04를 직접 실행하는 경우 아래 코드의 주석을 해제하세요.

In [ ]:
### 8.1 보안 액세스를 위한 AgentCore Identity CredentialsProvider 생성
## Lab-01 Cognito 설정에서 M2M 자격 증명 가져오기
# m2m_client_id = get_parameter(PARAMETER_PATHS['cognito']['m2m_client_id'])
# m2m_client_secret = get_parameter(PARAMETER_PATHS['cognito']['m2m_client_secret'])
# user_pool_id = get_parameter(PARAMETER_PATHS['cognito']['user_pool_id'])

## Cognito OIDC 검색 URL 구성
# discovery_url = f"https://cognito-idp.{AWS_REGION}.amazonaws.com/{user_pool_id}/.well-known/openid-configuration"

# AgentCore 클라이언트 초기화
# agentcore = boto3.client('bedrock-agentcore-control', region_name=AWS_REGION)

## OAuth2 Credential Provider 생성
## Secrets Manager에 M2M 자격 증명을 안전하게 저장
# credential_provider_response = agentcore.create_oauth2_credential_provider(
#      name='aiml301-m2m-credentials',
#      credentialProviderVendor='CustomOauth2',
#      oauth2ProviderConfigInput={
#          'customOauth2ProviderConfig': {
#              'clientId': m2m_client_id,
#              'clientSecret': m2m_client_secret,
#              'oauthDiscovery': {
#                  'discoveryUrl': discovery_url
#              }
#          }
#      }
#  )

## 응답 추출
# oauth2_provider_arn = credential_provider_response['credentialProviderArn']
# client_secret_arn = credential_provider_response['clientSecretArn']['secretArn']

## 다음 섹션에서 사용할 수 있도록 SSM에 저장
# put_parameter(PARAMETER_PATHS['lab_03']['oauth2_provider_arn'], oauth2_provider_arn)
# put_parameter(PARAMETER_PATHS['lab_03']['oauth2_secret_arn'], client_secret_arn)

# print("✅ OAuth2 Credential Provider created")
# print(f"\n📋 Credential Storage:")
# print(f"   Provider ARN: {oauth2_provider_arn}")
# print(f"   Secret ARN: {client_secret_arn}")
# print(f"   Location: AWS Secrets Manager (encrypted)")
# print(f"   Credentials: M2M client_id + client_secret")

### 8.2: OAuth2 Credential Provider 생성 및 Runtime 대상 등록

In [ ]:
# Parameter Store에서 M2M 자격 증명 가져오기
# SSM에서 구성 가져오기
gateway_id = get_parameter(PARAMETER_PATHS["lab_04"]["gateway_id"], region_name=AWS_REGION)
runtime_arn = get_parameter(PARAMETER_PATHS["lab_04"]["runtime_arn"], region_name=AWS_REGION)

# Lab-03에서 생성한 credentials provider를 사용하는 경우
oauth2_provider_arn = get_parameter(PARAMETER_PATHS["lab_03"]["oauth2_provider_arn"], region_name=AWS_REGION)
# Lab-03을 사용하지 않는 경우 아래 주석을 해제하여 새 credentials provider 생성
# oauth2_provider_arn = get_parameter(PARAMETER_PATHS['lab_04']['oauth2_provider_arn'], region_name=AWS_REGION)

resource_server_id = get_parameter(PARAMETER_PATHS["cognito"]["resource_server_identifier"], region_name=AWS_REGION)

In [ ]:
print(f"   Gateway ID: {gateway_id}")
print(f"   Gateway URL: {gateway_url}")
print(f"   Runtime ARN: {runtime_arn}")

In [ ]:
### M2M OAuth2를 사용하는 Runtime 대상 생성

import urllib.parse

# 중요: Runtime ARN에서 엔드포인트 URL 구성
# 형식: https://bedrock-agentcore.{region}.amazonaws.com/runtimes/{URL_ENCODED_ARN}/invocations?qualifier=DEFAULT
# ARN 예시: arn:aws:bedrock-agentcore:us-west-2:123456789012:runtime/my-runtime
encoded_arn = urllib.parse.quote(runtime_arn, safe="")
endpoint_url = (
    f"https://bedrock-agentcore.{AWS_REGION}.amazonaws.com/runtimes/{encoded_arn}/invocations?qualifier=DEFAULT"
)

# 세분화된 권한 부여를 위한 M2M 범위
m2m_scopes = [
    f"{resource_server_id}/mcp.invoke",
    f"{resource_server_id}/runtime.access",
]

# AgentCore 클라이언트 초기화
agentcore = boto3.client("bedrock-agentcore-control", region_name=AWS_REGION)

# Gateway 대상 생성 - M2M OAuth2를 사용하는 외부 MCP 서버로 Runtime 등록
# Gateway는 다음 작업을 자동으로 수행:
# 1. oauth2_provider_arn을 사용하여 Cognito에서 M2M 토큰 가져오기
# 2. Runtime 호출 시 Bearer 토큰 포함
# 3. Runtime에서 토큰을 검증하고 범위에 따라 작업 권한 부여
target_response = agentcore.create_gateway_target(
    gatewayIdentifier=gateway_id,
    name="aiml301-runtime-target",
    description="AgentCore Runtime with M2M OAuth2 authentication",
    targetConfiguration={
        "mcp": {
            "mcpServer": {
                "endpoint": endpoint_url  # ← URL 인코딩을 사용하여 Runtime ARN에서 구성
            }
        }
    },
    credentialProviderConfigurations=[
        {
            "credentialProviderType": "OAUTH",
            "credentialProvider": {
                "oauthCredentialProvider": {
                    "providerArn": oauth2_provider_arn,  # ← 섹션 9.1의 OAuth2 credential provider 참조
                    "scopes": m2m_scopes,
                }
            },
        }
    ],
)

target_id = target_response["targetId"]
put_parameter(
    PARAMETER_PATHS["lab_04"]["gateway_runtime_target"],
    target_id,
    region_name=AWS_REGION,
)

print("✅ Runtime added as Gateway target with M2M OAuth2")
print(f"   Target ID: {target_id}")
print(f"   Runtime ARN: {runtime_arn}")
print(f"   Endpoint: {endpoint_url}")
print(f"   Credential Provider: {oauth2_provider_arn}")

In [ ]:
import time

### 9.2 대상 상태 확인 및 동기화

# 대상이 READY 상태가 될 때까지 대기
print("\n⏳ Waiting for target to be READY...")
for attempt in range(30):
    target_info = agentcore.get_gateway_target(gatewayIdentifier=gateway_id, targetId=target_id)
    status = target_info.get("status", "UNKNOWN")

    if status == "READY":
        print("✅ Target is READY")
        break
    if status == "FAILED" or status == "SYNCHRONIZE_UNSUCCESSFUL":
        print(f"❌ Target in ERROR state: {target_info.get('statusReasons', 'No error message')}")
        break
    time.sleep(5)

# 도구 검색을 위해 동기화
agentcore.synchronize_gateway_targets(gatewayIdentifier=gateway_id, targetIdList=[target_id])

print("\n✅ Complete - Gateway will automatically manage M2M tokens")

  ##### 중요 참고 사항: 
  Gateway 역할에 필요한 권한:

  - bedrock-agentcore:GetResourceOauth2Token - credential provider에서 M2M 토큰 검색
  - secretsmanager:GetSecretValue - 저장된 OAuth2 자격 증명에 액세스

  이유: Gateway가 Runtime에 대한 MCP 연결을 시작하므로 OAuth2 M2M 자격 증명으로 인증해야 합니다.

  Runtime 역할에 필요한 권한:

  - bedrock-agentcore:GetWorkloadAccessToken - 워크로드 자격 증명 검증
  - bedrock-agentcore:CreateWorkloadIdentity - 자체 자격 증명 설정
  - bedrock-agentcore:GetResourceOauth2Token - Gateway에서 수신한 OAuth2 토큰 검증

  이유: Runtime은 OAuth2 Bearer 토큰이 포함된 MCP 요청을 Gateway에서 수신합니다. Runtime은 해당 토큰을 검증하고 수신 MCP 호출을 인증해야 합니다.

## 9. MCP Client를 통해 배포된 에이전트 테스트

**목표:** JWT 인증을 사용하여 Gateway를 통해 연결하고 배포된 에이전트가 작동하는지 확인합니다.

**접근 방식:** Cognito로 인증하고 MCPClient를 Gateway에 연결한 다음 예방 도구를 호출합니다.

**핵심 학습 내용:** MCP 프로토콜을 사용하여 프로덕션 에이전트를 테스트하는 방법을 알아봅니다.

### 9.1: Cognito로 인증하고 JWT 토큰 가져오기

In [ ]:
import boto3

# 테스트 사용자 자격 증명 가져오기
test_user_email = get_parameter(PARAMETER_PATHS["cognito"]["test_user_email"])
test_user_password = get_parameter(PARAMETER_PATHS["cognito"]["test_user_password"])

print("🔐 Authenticating with Cognito...")
print(f"   User: {test_user_email}")

cognito_client = boto3.client("cognito-idp", region_name=AWS_REGION)

# 사용자 인증
auth_response = cognito_client.initiate_auth(
    ClientId=user_auth_client_id,
    AuthFlow="USER_PASSWORD_AUTH",
    AuthParameters={"USERNAME": test_user_email, "PASSWORD": test_user_password},
)

access_token = auth_response["AuthenticationResult"]["AccessToken"]
print("✅ JWT token obtained")
print(f"   Token length: {len(access_token)} chars")

In [ ]:
print((access_token))

### 9.2: MCPClient 연결 및 도구 목록 조회

In [ ]:
from lab_helpers.lab_04.mcp_client import MCPClient

# MCP 클라이언트를 Gateway에 연결
print("🔌 Connecting to Gateway via MCPClient...")
client = MCPClient(gateway_url, access_token, timeout=300)

# MCP 세션 초기화
print("\n📡 Initializing MCP session...")
server_info = client.initialize()

# 사용 가능한 도구 나열
print("\n🔧 Listing available tools...")
tools = client.list_tools()

print(f"\n✅ Found {len(tools)} tools:")
for i, tool in enumerate(tools, 1):
    print(f"   {i}. {tool['name']}")
    desc = tool.get("description", "No description")
    first_line = desc.split("\n")[0][:80]
    print(f"      {first_line}...")

In [ ]:
# 모든 셀 실행 시나리오를 처리하기 위해 추가
import time

time.sleep(10)

### 9.3: 예방 분석 도구 호출

In [ ]:
# analyze_infrastructure_prevention 도구 호출
print("🔮 Invoking prevention_agent tool...")

result = client.call_tool(
    "aiml301-runtime-target___research_agent",
    {
        "research_topic_query": f"""Research based on this {additional_context} for best practices to minimize issues in the future."""
    },
)

print("\n✅ Prevention analysis complete:")
if not result["isError"]:
    print(result["content"][0]["text"])

else:
    print(f"   Error: {result}")

## 10. 정리

**목표:** Lab-04의 모든 AWS 리소스를 제거합니다.

**접근 방식:** cleanup_lab_04를 사용하여 Gateway, Runtime, IAM 역할 및 로그를 제거합니다.

**참고:** 지속적인 비용이 발생하지 않도록 테스트를 마친 후 실행합니다.

In [ ]:
# Lab-04의 모든 리소스 정리
print("🧹 Cleaning up Lab-04 resources...")
print("\nWARNING: This will delete:")
print("  • AgentCore Gateway and all targets")
print("  • AgentCore Runtime")
print("  • OAuth2 Credential Provider")
print("  • IAM roles")
print("  • CloudWatch logs")
print("\nTo proceed, uncomment the line below:")
print("# cleanup_lab_04(region_name=AWS_REGION, verbose=True)")

# 정리를 실행하려면 주석 해제:
# cleanup_lab_04(region_name='us-west-2', verbose=True)

## 요약: Lab 4 - 프로덕션 배포가 포함된 예방 에이전트

✅ **완료한 작업:**
1. ✓ Browser가 통합된 Strands 예방 에이전트 생성
2. ✓ Browser 웹 접근 및 콘텐츠 추출 테스트
3. ✓ 예방 분석 워크플로 실행(로컬)
4. ✓ 에이전트를 AgentCore Runtime에 배포
5. ✓ CloudWatch Logs 전송 구성
6. ✓ Cognito JWT 인증을 사용하는 Gateway 배포
7. ✓ OAuth2 M2M을 사용하여 Runtime을 MCP 대상으로 등록
8. ✓ JWT 인증으로 MCPClient를 통해 테스트

**아키텍처:**
```
Prevention Agent (Strands + Browser)
    ↓
AgentCore Runtime (FastMCP)
    ↓
Gateway (JWT auth inbound, OAuth2 M2M outbound)
    ↓
MCPClient (test via JWT)
```

**주요 기능:**
- **선제적 예방**: 인프라에서 문제를 예방할 기회를 분석
- **조사 기반**: AgentCore Browser를 사용하여 최신 AWS 모범 사례 조사
- **프로덕션 수준**: 서버리스, 자동 크기 조정, 관찰 가능
- **보안**: 이중 인증 계층(JWT + OAuth2)

**사용 가능한 MCP 도구 3개:**
1. `analyze_infrastructure_prevention` - 문제 예방 기회 조사
2. `research_aws_best_practices` - AWS 주제 조사
3. `validate_prevention_environment` - 환경 준비 상태 검증

**다음 단계:**
- Lab-05: 멀티 에이전트 오케스트레이션(문제 해결 + 예방)
- Lab-03과 Lab-04를 통합 SRE 자동화 파이프라인으로 결합
- 조사 → 예방 → 문제 해결 워크플로